# AudioAssistent: ручное тестирование
**Часть A. Локально** (без сервера и Postgres): аудио → транскрипт → говорящие → статус. Нужна только Ollama (и mlx-whisper для аудио).
**Часть B. Ручки сервиса** (нужны запущенные `uvicorn app.main:app` и PostgreSQL): загрузка записи, каждая стадия отдельно, свой промпт к модели, просмотр запуска, удаление.

Запись кладём в `data/audio/` под именем `YYYYMMDDHHMMSS.WAV` (из имени берутся дата и время встречи).

In [1]:
import sys, json
from pathlib import Path
from datetime import date

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if ".venv" not in sys.executable:
    raise RuntimeError(f"Ядро запущено не из .venv проекта ({sys.executable}). "
                       "Kernel -> Change Kernel -> «AudioAssistent (.venv)» и перезапустите ядро.")
sys.path.insert(0, str(ROOT))

from observability import init_observability
init_observability()                      # структурные логи; в ноутбуке удобно LOG_FORMAT=console

from audio_assistant.cli import build_local_pipeline   # Ollama + файловая история, без Postgres
from audio_assistant.client.pipeline import load_transcript
from audio_assistant.client.domain import Transcript
from audio_assistant.client.recordings import list_recordings, parse_recording_datetime

pipeline, settings = build_local_pipeline()
print("Аудитории:", pipeline.prompts.available())
print("Подсказка для Whisper:", pipeline.team.names_hint())

{"timestamp": "2026-11-09T09:03:35.376182Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "observability", "message": "[RUN_APP] [START] Application initialization"}
{"timestamp": "2026-11-09T09:03:35.377269Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "observability.tracing.setup", "message": "[STEP] Tracing initialized (no exporter)", "service_name": "rag-service"}
{"timestamp": "2026-11-09T09:03:35.377507Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "observability", "message": "[STEP] Observability initialized", "log_format": "json", "log_level": "INFO", "monitoring_enabled": true, "service_name": "rag-service", "tracing_enabled": true, "workspace": "dev"}


Аудитории: ['management', 'project_managers', 'team']
Подсказка для Whisper: Дейли. Участники: Гончар Роман, Губриенко Денис, Агафонов Алексей, Вертоградова Екатерина, Зеленский Павел, Абдуллин Равиль, Нестеренко Илья, Дронов Сергей, Тимофеев Сергей.


In [2]:
import subprocess, tempfile
from IPython.display import Audio, Image as IPImage, display
from audio_assistant.client.image_prep import prepare_image
from audio_assistant.client.transcription.mlx_backend import ensure_ffmpeg

def play_audio(path, start=0, seconds=None):
    """Плеер прямо в ноутбуке. WAV с диктофона большой (десятки МБ), поэтому для плеера он пережимается
    в моно mp3 64 кбит/с (на транскрибацию это не влияет: туда уходит исходный файл).
    start/seconds - проиграть только фрагмент, например play_audio(AUDIO, start=60, seconds=30)."""
    path = Path(path).expanduser()
    if not path.exists():
        print("нет файла:", path); return
    cmd = ["ffmpeg", "-v", "error", "-y", "-ss", str(start)]
    if seconds:
        cmd += ["-t", str(seconds)]
    tmp = Path(tempfile.mkdtemp()) / (path.stem + ".mp3")
    try:
        ensure_ffmpeg()
        subprocess.run(cmd + ["-i", str(path), "-ac", "1", "-b:a", "64k", str(tmp)], check=True)
        player = Audio(filename=str(tmp))
        note = f"{path.name} | {path.stat().st_size / 1e6:.1f} МБ | плеер: mp3 {tmp.stat().st_size / 1e6:.1f} МБ"
    except (RuntimeError, subprocess.CalledProcessError, OSError) as exc:
        print("ffmpeg не помог, плеер получит исходный файл:", exc)
        player, note = Audio(filename=str(path)), f"{path.name} | {path.stat().st_size / 1e6:.1f} МБ"
    print(note)
    display(player)

def show_images(paths, width=480):
    """Показывает фото так, как его получит модель: после поворота по EXIF и уменьшения до image_max_side."""
    max_side = getattr(settings, "image_max_side", 1568)
    for path in [Path(p).expanduser() for p in paths]:
        try:
            prepared = prepare_image(path.read_bytes(), max_side, path.name)
        except (OSError, ValueError) as exc:
            print(path.name, "-", exc); continue
        print(f"{path.name} | исходник {prepared.original_bytes / 1e6:.1f} МБ -> модели уйдёт "
              f"{prepared.width}x{prepared.height}, {len(prepared.data) / 1e6:.2f} МБ")
        display(IPImage(data=prepared.data, width=min(width, prepared.width)))


# Часть A. Локально

## A1. Записи и дата встречи

In [3]:
recordings = list_recordings(settings.audio_dir)
for r in recordings:
    print(r["file_name"], r["size_bytes"], "байт | время записи:", r["recorded_at"])
AUDIO = settings.audio_dir / recordings[0]["file_name"] if recordings else None
DAY = (parse_recording_datetime(AUDIO.name).date() if AUDIO and parse_recording_datetime(AUDIO.name) else date.today())
print("Берём:", AUDIO, "| дата встречи:", DAY)
if AUDIO:
    play_audio(AUDIO)


20261109110122.WAV 28033536 байт | время записи: 2026-11-09 11:01:22
20261102103215.WAV 11993600 байт | время записи: 2026-11-02 10:32:15
Берём: /Users/dmitry/Desktop/GitHub/AudioAssistent/data/audio/20261109110122.WAV | дата встречи: 2026-11-09
20261109110122.WAV | 28.0 МБ | плеер: mp3 7.0 МБ


## A2. Аудио → сырой транскрипт (Whisper). Результат кэшируется, чтобы не транскрибировать заново

In [4]:
%%time
FORCE = False   # True — перетранскрибировать
cache = settings.output_dir / f"{AUDIO.stem}.raw.json" if AUDIO else None
if AUDIO and cache.exists() and not FORCE:
    raw = Transcript.from_dict(json.loads(cache.read_text(encoding="utf-8")))
    print("из кэша:", cache.name)
elif AUDIO:
    settings.output_dir.mkdir(parents=True, exist_ok=True)
    raw = pipeline.transcribe(AUDIO)
    cache.write_text(json.dumps(raw.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
else:
    raw = load_transcript(ROOT / "examples" / "sample_transcript.txt")   # запасной вариант без аудио
    print("аудио нет, берём examples/sample_transcript.txt")
for s in raw.segments:
    print(f"{s.start:6.1f}-{s.end:6.1f}  {s.text}")

{"timestamp": "2026-11-09T09:04:16.195816Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/agent-harnesses \"HTTP/1.1 200 OK\""}
{"timestamp": "2026-11-09T09:04:16.353081Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/mlx-community/whisper-large-v3-turbo/revision/main \"HTTP/1.1 200 OK\""}


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

   0.0-   2.0  Так, ну давайте тогда начинать, Рома.
   4.0-   7.0  Дима. С тебя давай. Давай, давай, давай.
   8.0-  19.0  Так, ну я в пятницу получил от Дениса исходные джессончики, получается с извлечением контента из документов после OCR.
  20.0-  29.0  Вот, на этих джессонах я сейчас перепрогоняю свой pipeline, вот под них его там дорабатывал, вот, и, соответственно,
  30.0-  32.2  добежит, буду подбивать метрики.
  32.9-  33.6  Но кажется, что
  33.6-  36.0  еще лучше должно
  36.0-  38.3  стать по метрикам.
  39.4-  40.3  Потому что
  40.3-  41.8  меньше шума,
  42.1-  43.5  приходится закидывать в ламку,
  44.1-  45.1  и лучше
  45.1-  47.8  реальный результат
  47.8-  48.8  за счет этого.
  49.8-  52.3  Потому что, первое, это порядок страниц,
  52.5-  54.2  он больше не битый, как был до этого.
  55.3-  56.6  И, соответственно,
  56.7-  57.2  всякие там
  57.2-  59.0  техническая история
  59.0-  61.4  с информацией
  61.4-  62.6  о письме в банк,
  63.6-  64.9  с информацией

## A3. Разметка говорящих
Два метода: **llm** (модель размечает чанками; имена распознаёт код, модель решает, передаёт ли упоминание слово) и **rules** (регулярные выражения). `— ведущий —` — речь вне меток.

In [5]:
METHOD = "llm"          # llm | rules
attributed = pipeline.attribute(raw, METHOD)
print("метод:", attributed.meta.get("method"), "| чанков:", attributed.meta.get("chunks"),
      "| чанки, размеченные правилами из-за сбоя модели:", attributed.meta.get("fallback_chunks"))
print(attributed.render())

{"timestamp": "2026-11-09T09:05:24.378905Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:05:42.126466Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 649}
{"timestamp": "2026-11-09T09:05:42.136453Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:05:48.775272Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 115}
{"timestamp": "2026-11-09T09:05:48.778608Z

метод: llm | чанков: 2 | чанки, размеченные правилами из-за сбоя модели: []
[???] Так, ну давайте тогда начинать, Рома.
[Гончар Роман] Дима. С тебя давай. Давай, давай, давай. Так, ну я в пятницу получил от Дениса
[Губриенко Денис] исходные джессончики, получается с извлечением контента из документов после OCR. Вот, на этих джессонах я сейчас перепрогоняю свой pipeline, вот под них его там дорабатывал, вот, и, соответственно, добежит, буду подбивать метрики. Но кажется, что еще лучше должно стать по метрикам. Потому что меньше шума, приходится закидывать в ламку, и лучше реальный результат за счет этого. Потому что, первое, это порядок страниц, он больше не битый, как был до этого. И, соответственно, всякие там техническая история с информацией о письме в банк, с информацией о подписях электронных, она уже не шумит. И текущий по конкретному основному содержанию файла пробегается. Потом, если от него сейчас додатно считается, метрики обновлю еще раз на страничке. И, наверное, пока эту и

### Сравнение методов: чья реплика чья

In [6]:
by_llm = pipeline.attribute(raw, "llm").text_by_speaker()
by_rules = pipeline.attribute(raw, "rules").text_by_speaker()
print(f"{'говорящий':28} {'llm':>5} {'rules':>6}   (слов)")
for who in dict.fromkeys([*by_llm, *by_rules]):
    print(f"{(who or '— ведущий —'):28} {len(by_llm.get(who, '').split()):>5} {len(by_rules.get(who, '').split()):>6}")

{"timestamp": "2026-11-09T09:07:09.394194Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:07:16.431365Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 649}
{"timestamp": "2026-11-09T09:07:16.438989Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:07:17.790410Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 115}
{"timestamp": "2026-11-09T09:07:17.791920Z

говорящий                      llm  rules   (слов)
— ведущий —                      6    317
Гончар Роман                    15      0
Губриенко Денис                298      0
Агафонов Алексей                 5      2
Вертоградова Екатерина          71     69
Зеленский Павел                205    207
Абдуллин Равиль                171      0
Нестеренко Илья                243    240
Дронов Сергей                  235      0
Тимофеев Сергей                306    708


In [7]:
spoken = attributed.text_by_speaker()
print("Без реплик:", [p.full_name for p in pipeline.team.participants if p.full_name not in spoken] or "нет")

Без реплик: нет


## A4. Статус для аудитории
`save=False` — не писать историю. Правьте `prompts/<аудитория>/` и перезапускайте ячейку.

In [8]:
AUDIENCE = "management"      # management | project_managers | team
res = pipeline.run_transcript(attributed, AUDIENCE, DAY, save=False)
print(res.text)

{"timestamp": "2026-11-09T09:07:30.962536Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:07:33.417314Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 81}
{"timestamp": "2026-11-09T09:07:33.426162Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T09:07:37.922061Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 200}
{"timestamp": "2026-11-09T09:07:37.924237Z"

Дейли 09.11.2026
Проект 3924:
1. Гончар Роман - нет данных о результатах работы, задача не выполнена, требуется уточнение планов.
2. Губриенко Денис - обновил метрики для проекта 3924 после фильтрации китайских классов и конфиденциальных данных, достигнув значения выше 83,5%, далее обучит модель на оставшихся 139 классах с малым количеством сэмплов.
Проект 3641:
1. Агафонов Алексей - нет данных о работе, сотрудник в отпуске.
2. Вертоградова Екатерина - выполнил разметку выборки ИП-шников с выделением отраслей КВЭНом для исходящих платежей самозанятых как физиков, оставив приоритет назначениям платежей; далее планирует пристроить оставшихся самозанятых и обучить классификатор на эмбеддингах.
3. Зеленский Павел - в отпуске, ожидание закрытия ЗМИ от поддержки до декабря для миграции источников в витрину, ПН не указан.
4. Абдуллин Равиль - собрал факторы Новые, обучил модель (точка 05), поставил пересборку транзакции на холдауте для скоринга, код готов к сдаче в пятницу с подготовкой докум

## A5. Один сотрудник, промпт целиком (отладка)

In [9]:
from string import Template
profile = pipeline.prompts.get(AUDIENCE)
who = next(p for p in pipeline.team.participants if p.full_name in spoken)
user = Template(profile.person_template).safe_substitute(
    date=DAY.strftime("%d.%m.%Y"), project_id=who.project_id, full_name=who.full_name,
    previous_status="нет данных", examples=profile.examples, utterance=spoken[who.full_name])
print(profile.system, "\n--------\n", user)

Ты помощник тимлида. Превращаешь расшифровку устной речи сотрудника на дейли в статус для РУКОВОДСТВА.

Правила:
- Один абзац в одну строку, 1–3 предложения, по-русски, без markdown и списков.
- Главное: что сделано (результат), что в процессе, что дальше, риски и проблемы. Мелкие технические детали опускай.
- Сохраняй названия систем, проектов, метрик и числа точно как в расшифровке. Не выдумывай факты, цифры и сроки.
- Игнорируй слова-паразиты, обращения и реплики не по делу.
- Стиль живой деловой, как в примерах: допустимы сокращения ("ПСИ", "ПН", "вт"), краткие обороты, "+ также ...".
- Если сказано, что нечего добавить, так и напиши коротко и добавь важные замечания, если они есть.
- Названия, похожие на искажённые распознаванием, исправляй только если уверен.
- СТРОГО: используй только то, что прямо сказано в расшифровке. Ничего не додумывай и не добавляй выводов, прогнозов, оценок и общих фраз ("дальнейшие шаги не указаны", "требует внимания", "нет данных", "ПСИ нет").
- Если о 

## A6. Свой промпт к записи (локально)
Для записей не по дейли: любой промпт поверх транскрипта. `{transcript}` подставляется в текст, нет метки — текст идёт в конец.
`dialogue` (с метками говорящих) берите после A3.

In [10]:
PROMPT = """Выпиши принятые решения и задачи (кто, что, срок). Ничего не домысливай.

Запись:
{transcript}"""
SOURCE = "raw"          # raw | dialogue

text = raw.text if SOURCE == "raw" else attributed.render()
result = pipeline.run_prompt(PROMPT, transcript_text=text, system="Ты ассистент по протоколам встреч.")
print(f"{result.model} | {result.duration_ms / 1000:.1f} с\n")
print(result.text)

{"timestamp": "2026-11-09T11:21:02.747300Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama request", "messages": 2, "model": "qwen3.5:4b"}
{"timestamp": "2026-11-09T11:21:25.045579Z", "level": "info", "service_id": "rag-service", "host": "MacBook-Pro-Dmitry.local", "logger": "ollama_client.client.ollama_client", "message": "Ollama response received", "content_length": 1727}


qwen3.5:4b | 22.3 с

Вот принятые решения и задачи из записи:

**Задачи:**

| Исполнитель | Описание задачи | Срок |
| :--- | :--- | :--- |
| **Рома** | Перепрогнать свой pipeline на новые джессончики (с извлечением контента после OCR), довести до конца, подбить метрики. Оставить текущую историю по основному содержанию файла. | Не указан (в рамках текущего этапа) |
| **Дима** | Копать историю по задачке сервисам 3478: понять, почему встретил DR и не прошло DR-тестирование (исходно обновления в S3 были в июне-июле). | Не указан |
| **Ваня** | Решить проблему с 139 классами, где мало сэмплов для моделирования. Направить им письма по этому вопросу. Обновить метрику после удаления китайских классов и конфиденциальной информации (не принимаемых на CDP). Проводить эксперименты. | Не указан |
| **Катя** | Пристроить оставшихся самозанятых к текущей выборке ИП-шников. Попробовать обучить классификатор на эмбеддингах своих. | Не указан |
| **Паша** | Доскоринг менеджеров (физлица). После проско

# Часть B. Ручки сервиса
Запуск: `POSTGRES_PORT=… uvicorn app.main:app --port 8005`. Каждая стадия сохраняет результат в Postgres под `run_id`,
повторный вызов заменяет его, поэтому можно править алиасы/промпты и перепроверять только нужный шаг.

In [11]:
import requests
BASE = "http://localhost:8005"

def api(method, path, **kw):
    r = requests.request(method, BASE + path, timeout=1800, **kw)
    try:
        body = r.json()
    except ValueError:
        body = r.text
    print(method, path, "->", r.status_code)
    return body

def show(obj, limit=3000):
    print(json.dumps(obj, ensure_ascii=False, indent=2)[:limit])

show(api("GET", "/assistant/health"))
show(api("GET", "/ollama/health"))
show(api("GET", "/postgres/health"))

GET /assistant/health -> 200
{
  "status": "ok",
  "history_backend": "postgres",
  "transcriber_backend": "faster_whisper",
  "stages_available": true,
  "audiences": [
    "management",
    "project_managers",
    "team"
  ],
  "participants": 9
}
GET /ollama/health -> 200
{
  "status": "ok",
  "url": "http://host.docker.internal:11434",
  "model": "qwen3.5:4b",
  "model_available": true
}
GET /postgres/health -> 200
{
  "status": "ok",
  "host": "postgres",
  "port": 5432,
  "database": "audio"
}


### Проверка моделей и зависимостей
Та же проверка выполняется при старте сервиса (лог `[CHECK] ...`). Здесь её можно повторить в любой момент:
Ollama и её модель, Whisper (файлы модели в кэше), ffmpeg, Postgres, папки, промпты по типам встреч.
Статус `loading` — модель Whisper докачивается в фоне, `warn` — работать можно, `error` — не заработает.

In [ ]:
models = api("GET", "/assistant/models")
print("готово к работе:", models["ready"])
for chk in models["checks"]:
    print(f"  {chk['status']:8} {chk['name']:12} {chk['detail']}")

## B1. Записи в `data/audio`

In [12]:
files = api("GET", "/assistant/audio")
show(files)
FILE = files[0]["file_name"]
play_audio(settings.audio_dir / FILE)   # плеер; фрагмент: play_audio(path, start=60, seconds=30)


GET /assistant/audio -> 200
[
  {
    "file_name": "20261102103215.WAV",
    "size_bytes": 11993600,
    "recorded_at": "2026-11-02T10:32:15"
  }
]


## B1a. Загрузка записи через ручку
`POST /assistant/audio` кладёт файл в `data/audio` под его именем (дата берётся из имени `YYYYMMDDHHMMSS`).
Тот же файл не перезаписывается (409), пока не передан `overwrite=true`. С `transcribe=true` сразу запускается стадия 1.
Транскрибацию можно вызвать и отдельно: B2.

In [26]:
UPLOAD = None          # например Path("~/Downloads/20261105101500.wav").expanduser()
if UPLOAD:
    play_audio(UPLOAD)   # прослушать до загрузки
    with open(UPLOAD, "rb") as fh:
        r = requests.post(BASE + "/assistant/audio", files={"file": (UPLOAD.name, fh)},
                          data={"overwrite": "false", "transcribe": "false", "kind": "daily", "tags": ""}, timeout=1800)
    print(r.status_code); show(r.json())
    FILE = UPLOAD.name

## B2. Стадия 1: транскрибация (создаёт `run_id`)
Запрос возвращается сразу (`wait=False`), Whisper работает в фоне, а ячейка показывает прогресс. Если прервать ячейку или нажать её
повторно, второй транскрибации не будет: вернётся тот же `run_id` (`already_running`). Файлы идут по очереди: Whisper занимает все ядра.
На CPU в Docker ориентир ~0,4 от длины записи (на Mac через Metal быстрее); при первом запуске ещё скачивается модель (~1,5 ГБ).

`KIND` и `TAGS` записываются в Postgres вместе с запуском (позже их можно поменять: B6a). Типы встреч: `config/meeting_kinds.yaml`.


In [ ]:
import time

def wait_run(run_id, poll=5, timeout=3 * 3600):
    """Ждёт конца транскрибации и печатает прогресс. Ctrl+C останавливает ожидание, а не саму транскрибацию."""
    started = time.time()
    while time.time() - started < timeout:
        run = requests.get(f"{BASE}/assistant/runs/{run_id}", timeout=30).json()["run"]
        if run["status"] == "error":
            raise RuntimeError(run["error"])
        if run["status"] != "transcribing":
            print(f"\nготово за {time.time() - started:.0f} с, стадия: {run['status']}")
            return
        pct = f"{run['progress']:.0%}" if run.get("progress") is not None else "модель загружается / прогресс неизвестен"
        print(f"\r{time.time() - started:5.0f} с  транскрибация: {pct}      ", end="")
        time.sleep(poll)
    raise TimeoutError("транскрибация идёт дольше таймаута; запуск не потерян, смотрите B6")

KIND = "daily"                  # daily | one_to_one | business_customer | management | other (config/meeting_kinds.yaml)
TAGS = []                       # любые пометки, например ["проект 3924", "срочно"]

play_audio(settings.audio_dir / FILE)   # прослушать запись перед транскрибацией
t = api("POST", "/assistant/transcribe", json={"file_name": FILE, "wait": False, "kind": KIND, "tags": TAGS})
RUN = t["run_id"]
DAY = date.fromisoformat(t["meeting_date"])
print("run_id =", RUN, "| дата встречи:", t["meeting_date"], "| записано:", t["recorded_at"],
      "| уже шла" if t["already_running"] else "")
wait_run(RUN)
for s in api("GET", f"/assistant/runs/{RUN}")["raw_transcript"]:
    print(f"{s['start']:6.1f}-{s['end']:6.1f}  {s['text']}")

## B3. Стадия 2: разбивка по говорящим
Метод можно переключать и вызывать стадию повторно, чтобы сравнить: транскрибация не повторяется.

In [15]:
METHOD = "llm"          # llm | rules | None (из настроек сервиса)
a = api("POST", f"/assistant/runs/{RUN}/attribute", json={"method": METHOD})
print("метод:", a["method"], "| чанков:", a["chunks"], "| размечено правилами из-за сбоя модели:", a["fallback_chunks"])
for u in a["dialogue"]:
    print(f"[{u['speaker'] or '— ведущий —'}] {u['text']}")
print("Без реплик:", a["without_utterances"])

POST /assistant/runs/1/attribute -> 200
метод: llm | чанков: 1 | размечено правилами из-за сбоя модели: []
[— ведущий —] Так, всем привет. Видимо, не все услышали, увидели, но окей, начнем тогда просто в 1 часов. Давайте подключимся Васю поздравить к Артему. Поэтому начнем тогда текущим составом. Так, Леш,
[Агафонов Алексей] Алексей, давай. Да, что у меня, у меня на повестке дня получается ПСИ сегодня, вот, сейчас все подготавливаю, все файлики, все открываю заранее, просто надо сейчас все это показать, будет с этой снов задач когда-то. Ты же понял, да, что там по новой задаче, по сути, до какого-то там пятого-шестого пункта, это просто идет правильная свертка таблиц, для того, чтобы можно было потом уже посчитать. Только последние пару пунктов показывают как раз все кейсы и что вообще нужно сделать. Ну, я пока иду по этому, то есть смотрю вот эти вот функции выполняемые. Ну, то есть я понял, что сначала собирается вот эта вот таблица, ну, как пример, вот эта вот таблица и дальше вот ч

## B4. Стадия 3: статусы по людям (LLM)

In [16]:
AUD = "management"
s = api("POST", f"/assistant/runs/{RUN}/summarize", json={"audience": AUD})
for st in s["statuses"]:
    if st["has_data"]:
        print("-", st["full_name"], ":", st["text"])

POST /assistant/runs/1/summarize -> 200
- Агафонов Алексей : ПСИ сегодня, свертка таблиц идет для последующего расчета, протокол согласуют сутки, завтра напишет Сергею.
- Вертоградова Екатерина : Катя начала готовить таргет для модели классификации вида деятельности самозанятых на основе АКВД, Пешников и отраслей, выделенных ЛЛМкой; план по приведению данных к единообразию утвержден, дальнейшие шаги не указаны.
- Абдуллин Равиль : нет данных о результатах работы, в процессе попытки запуска кода по циклу месяцев, документация не завершена, риски связаны с непониманием логики обработки промежутков.
- Нестеренко Илья : нет данных о выполнении задач, запланировано завершение работ по кассовому разрыву и документации, требуется обсуждение приоритетов с Сережей.
- Дронов Сергей : в четверг данные будут после триггера ДАК и репликации, интеграция работает по графику, завтра согласование для запуска в работу ЗНИ.


## B5. Стадия 4: итоговый отчёт (без LLM)

In [17]:
r = api("POST", f"/assistant/runs/{RUN}/report", json={"audience": AUD})
print(r["text"])

POST /assistant/runs/1/report -> 200
Дейли 02.11.2026
Проект 3924:
1. Гончар Роман - нет обновлений
2. Губриенко Денис - нет обновлений
Проект 3641:
1. Агафонов Алексей - ПСИ сегодня, свертка таблиц идет для последующего расчета, протокол согласуют сутки, завтра напишет Сергею.
2. Вертоградова Екатерина - Катя начала готовить таргет для модели классификации вида деятельности самозанятых на основе АКВД, Пешников и отраслей, выделенных ЛЛМкой; план по приведению данных к единообразию утвержден, дальнейшие шаги не указаны.
3. Зеленский Павел - нет обновлений
4. Абдуллин Равиль - нет данных о результатах работы, в процессе попытки запуска кода по циклу месяцев, документация не завершена, риски связаны с непониманием логики обработки промежутков.
5. Нестеренко Илья - нет данных о выполнении задач, запланировано завершение работ по кассовому разрыву и документации, требуется обсуждение приоритетов с Сережей.
6. Дронов Сергей - в четверг данные будут после триггера ДАК и репликации, интеграци

## B6. Всё по запуску и список запусков

In [18]:
d = api("GET", f"/assistant/runs/{RUN}")
print("стадия:", d["run"]["status"], "| сырых сегментов:", len(d["raw_transcript"]),
      "| реплик:", len(d["dialogue"]), "| аудитории:", list(d["statuses"]), "| отчёты:", list(d["reports"]))
show(api("GET", "/assistant/runs?limit=5"))

GET /assistant/runs/1 -> 200
стадия: reported | сырых сегментов: 75 | реплик: 6 | аудитории: ['management'] | отчёты: ['management']
GET /assistant/runs?limit=5 -> 200
[
  {
    "id": 1,
    "meeting_date": "2026-11-02",
    "recorded_at": "2026-11-02T10:32:15",
    "source": "20261102103215.WAV",
    "status": "reported",
    "error": null,
    "created_at": "2026-11-02T09:33:25.907556+00:00",
    "updated_at": "2026-11-02T10:04:04.329840+00:00"
  }
]


## B6a. Тип встречи и теги: правка и фильтры
`PATCH /assistant/runs/{id}` меняет только переданные поля (`{"kind": null}` снимает тип). В списке запусков можно фильтровать
по типу, тегу и периоду: `GET /assistant/runs?kind=&tag=&date_from=&date_to=`.

In [ ]:
import pandas as pd

show(api("GET", "/assistant/prompts"))          # типы встреч, есть ли для них промпт, сводки, аудитории

api("PATCH", f"/assistant/runs/{RUN}", json={"kind": "daily", "tags": ["проект 3924"]})   # любое из полей можно опустить

def runs_df(**filters):
    r = api("GET", "/assistant/runs", params={"limit": 1000, **filters})
    return pd.DataFrame(r, columns=["id", "meeting_date", "kind", "tags", "status", "source", "error"])

display(runs_df())                                                   # все
display(runs_df(kind="daily", date_from="2026-11-01", date_to="2026-11-30"))
display(runs_df(tag="проект 3924"))

## B6b. Разбор встречи промптом её типа
Промпты пишете вы: `prompts/meetings/<тип>/system.md` и `user.md` (метки `{transcript}`, `{date}`, `{participants}` и др. — в `prompts/meetings/README.md`).
`POST /assistant/runs/{id}/analyze` берёт промпт по типу встречи (или `prompt` явно), результат сохраняется в Postgres (`analyses`)
и виден в `GET /assistant/runs/{id}`. Файлы читаются на каждый запрос: правьте промпт и повторяйте вызов.
`source`: `dialogue` — с разметкой говорящих (нужна стадия B3), `raw` — как расслышал Whisper.

In [ ]:
an = api("POST", f"/assistant/runs/{RUN}/analyze", json={"source": "dialogue"})          # prompt по умолчанию = тип встречи
# an = api("POST", f"/assistant/runs/{RUN}/analyze", json={"prompt": "management", "source": "raw"})   # другой промпт
print(f"{an['prompt']} | {an['model']} | частей: {an['chunks']}\n")
print(an["text"])

## B7. Сквозные запуски
`/assistant/runs/{id}/process` — стадии 2–4 по уже сохранённому транскрипту; `/assistant/process-file` — всё одним вызовом.

In [19]:
# r = api("POST", f"/assistant/runs/{RUN}/process", json={"audience": "team"})
# r = api("POST", "/assistant/process-file", json={"file_name": FILE, "audience": AUD})

## B7a. Таблицы Postgres как DataFrame
`GET /assistant/tables` — сколько строк в каждой таблице, `GET /assistant/tables/{имя}?run_id=&limit=` — строки (свежие первыми).
Таблицы: `meeting_runs` (запуски), `transcripts` (сырой/размеченный транскрипт), `speaker_utterances` (реплики по говорящим),
`person_statuses` (статусы по людям), `reports` (итоговые отчёты), `llm_calls` (журнал вызовов модели), `analyses` (разборы встреч по типам), `digests` (сводки за период).

In [20]:
import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

def table(name, run_id=None, limit=100) -> pd.DataFrame:
    r = api("GET", f"/assistant/tables/{name}", params={"run_id": run_id, "limit": limit})
    print(f"{name}: показано {len(r['rows'])} из {r['total']}")
    return pd.DataFrame(r["rows"], columns=r["columns"])

display(pd.Series(api("GET", "/assistant/tables")["counts"], name="строк").to_frame())

GET /assistant/tables -> 200


,строк
llm_calls,6
meeting_runs,1
person_statuses,9
reports,1
speaker_utterances,6
transcripts,2


In [21]:
runs = table("meeting_runs")            # все запуски: стадия, ошибка, время
display(runs)

GET /assistant/tables/meeting_runs -> 200
meeting_runs: показано 1 из 1


,id,meeting_date,recorded_at,source,status,error,created_at,updated_at
0,1,2026-11-02,2026-11-02T10:32:15,20261102103215.WAV,reported,None,2026-11-02T09:33:25.907556Z,2026-11-02T10:04:04.329840Z


In [22]:
# по одному запуску (RUN из B2); run_id=None — все запуски
display(table("speaker_utterances", RUN)[["position", "full_name", "start_sec", "end_sec", "text"]])
display(table("person_statuses", RUN)[["audience", "full_name", "project_id", "has_data", "text"]])
display(table("reports", RUN)[["audience", "text"]])

GET /assistant/tables/speaker_utterances -> 200
speaker_utterances: показано 6 из 6


,position,full_name,start_sec,end_sec,text
0,5,Нестеренко Илья,242.79,373.02,"с выходом в отпуск к тебе. Я учитель на тех, и к Диону, и к команде. Наверное, задача, вот, надо будет обсудить с Се..."
1,4,Абдуллин Равиль,183.68,244.67,"Вера, вот этот код, кажется, на все промежутки было написано для, как бы, для... Сейчас попробую просто запустить сн..."
2,3,Вертоградова Екатерина,150.88,183.68,"Катя. Да, привет. У меня определение вида деятельности самозанятых. Начинаю готовить таргет для модели классификации..."
3,2,Дронов Сергей,109.73,152.82,"чтобы он, ну, как будет согласован, чтобы он их пускал уже в работу. Ближайшую дату для ЗНИ, если завтра он их пусти..."
4,1,Агафонов Алексей,0.00,117.31,"Алексей, давай. Да, что у меня, у меня на повестке дня получается ПСИ сегодня, вот, сейчас все подготавливаю, все фа..."
5,0,NaN,0.00,19.00,"Так, всем привет. Видимо, не все услышали, увидели, но окей, начнем тогда просто в 1 часов. Давайте подключимся Васю..."


GET /assistant/tables/person_statuses -> 200
person_statuses: показано 9 из 9


,audience,full_name,project_id,has_data,text
0,management,Тимофеев Сергей,3641,False,нет обновлений
1,management,Дронов Сергей,3641,True,"в четверг данные будут после триггера ДАК и репликации, интеграция работает по графику, завтра согласование для запу..."
2,management,Нестеренко Илья,3641,True,"нет данных о выполнении задач, запланировано завершение работ по кассовому разрыву и документации, требуется обсужде..."
3,management,Абдуллин Равиль,3641,True,"нет данных о результатах работы, в процессе попытки запуска кода по циклу месяцев, документация не завершена, риски ..."
4,management,Зеленский Павел,3641,False,нет обновлений
5,management,Вертоградова Екатерина,3641,True,"Катя начала готовить таргет для модели классификации вида деятельности самозанятых на основе АКВД, Пешников и отрасл..."
6,management,Агафонов Алексей,3641,True,"ПСИ сегодня, свертка таблиц идет для последующего расчета, протокол согласуют сутки, завтра напишет Сергею."
7,management,Губриенко Денис,3924,False,нет обновлений
8,management,Гончар Роман,3924,False,нет обновлений


GET /assistant/tables/reports -> 200
reports: показано 1 из 1


,audience,text
0,management,Дейли 02.11.2026\nПроект 3924:\n1. Гончар Роман - нет обновлений\n2. Губриенко Денис - нет обновлений\nПроект 3641:\...


In [23]:
calls = table("llm_calls", RUN)          # журнал вызовов модели: промпты, ответы, длительность
display(calls.drop(columns=["system_prompt", "user_prompt", "response"], errors="ignore"))

GET /assistant/tables/llm_calls -> 200
llm_calls: показано 6 из 6


,id,run_id,full_name,audience,model,status,error,duration_ms,created_at
0,6,1,Дронов Сергей,management,qwen3.5:4b,success,None,2313.66,2026-11-02T10:04:01.248986Z
1,5,1,Нестеренко Илья,management,qwen3.5:4b,success,None,2576.98,2026-11-02T10:03:58.921867Z
2,4,1,Абдуллин Равиль,management,qwen3.5:4b,success,None,2151.63,2026-11-02T10:03:56.339472Z
3,3,1,Вертоградова Екатерина,management,qwen3.5:4b,success,None,4241.65,2026-11-02T10:03:54.178464Z
4,2,1,Агафонов Алексей,management,qwen3.5:4b,success,None,4059.87,2026-11-02T10:03:49.928723Z
5,1,1,NaN,,qwen3.5:4b,success,None,14423.02,2026-11-02T10:03:35.693539Z


## B8. Удаление
Удаляются запуск и все связанные строки (транскрипты, реплики, статусы, отчёты, журнал LLM); статусы удалённой встречи
не попадают в «прошлый статус». **По умолчанию функция только считает** (`dry_run`) и показывает, сколько строк
в каких таблицах будет удалено; по-настоящему удаляет только с `confirm=True`.

In [24]:
def delete_runs(*, run_id=None, file_name=None, meeting_date=None, everything=False, confirm=False):
    """Один способ выбора за раз: run_id | file_name и/или meeting_date | everything."""
    params = {"dry_run": not confirm}
    if everything:
        params["all"] = True
    if file_name:
        params["file_name"] = file_name
    if meeting_date:
        params["meeting_date"] = str(meeting_date)
    path = f"/assistant/runs/{run_id}" if run_id else "/assistant/runs"
    result = api("DELETE", path, params=params)
    print("УДАЛЕНО" if confirm else "пробный прогон (ничего не удалено)", "->", result)
    return result

delete_runs(run_id=RUN)                          # по id
delete_runs(file_name=FILE)                      # по имени файла
delete_runs(meeting_date=DAY)                    # по дате встречи
delete_runs(everything=True)                     # ВСЕ таблицы целиком

DELETE /assistant/runs/1 -> 200
пробный прогон (ничего не удалено) -> {'dry_run': True, 'run_ids': [1], 'deleted': {'llm_calls': 6, 'person_statuses': 9, 'reports': 1, 'speaker_utterances': 6, 'transcripts': 2, 'meeting_runs': 1}}
DELETE /assistant/runs -> 200
пробный прогон (ничего не удалено) -> {'dry_run': True, 'run_ids': [1], 'deleted': {'llm_calls': 6, 'person_statuses': 9, 'reports': 1, 'speaker_utterances': 6, 'transcripts': 2, 'meeting_runs': 1}}
DELETE /assistant/runs -> 200
пробный прогон (ничего не удалено) -> {'dry_run': True, 'run_ids': [1], 'deleted': {'llm_calls': 6, 'person_statuses': 9, 'reports': 1, 'speaker_utterances': 6, 'transcripts': 2, 'meeting_runs': 1}}
DELETE /assistant/runs -> 200
пробный прогон (ничего не удалено) -> {'dry_run': True, 'run_ids': [], 'deleted': {'llm_calls': 6, 'person_statuses': 9, 'reports': 1, 'speaker_utterances': 6, 'transcripts': 2, 'meeting_runs': 1}}


{'dry_run': True,
 'run_ids': [],
 'deleted': {'llm_calls': 6,
  'person_statuses': 9,
  'reports': 1,
  'speaker_utterances': 6,
  'transcripts': 2,
  'meeting_runs': 1}}

In [25]:
CONFIRM_DELETE = True      # True — удалить по-настоящему, раскомментировав нужную строку
if CONFIRM_DELETE:
    delete_runs(run_id=RUN, confirm=True)
    # delete_runs(file_name=FILE, confirm=True)
    # delete_runs(meeting_date=DAY, confirm=True)
    # delete_runs(everything=True, confirm=True)       # очищает ВСЕ таблицы
    display(pd.Series(api("GET", "/assistant/tables")["counts"], name="строк").to_frame())

DELETE /assistant/runs/1 -> 200
УДАЛЕНО -> {'dry_run': False, 'run_ids': [1], 'deleted': {'llm_calls': 6, 'person_statuses': 9, 'reports': 1, 'speaker_utterances': 6, 'transcripts': 2, 'meeting_runs': 1}}
GET /assistant/tables -> 200


,строк
llm_calls,0
meeting_runs,0
person_statuses,0
reports,0
speaker_utterances,0
transcripts,0


## B9. Свой промпт к модели (не дейли)
`POST /assistant/llm` берёт транскрипт запуска и применяет к нему ваш промпт; статусы и отчёты не трогает.
- `{transcript}` в промпте заменяется текстом; нет метки → транскрипт дописывается в конец.
- `source`: `raw` — как расслышал Whisper (стадии attribute не нужно); `dialogue` — с метками говорящих (нужна стадия 2).
- `system` и `json_schema` необязательны; вызов пишется в журнал `llm_calls` с `audience="custom"`.

In [ ]:
PROMPT = """Ты помогаешь вести протокол. Выпиши из записи:
1. Принятые решения.
2. Задачи: кто, что, срок (если назван).
Ничего не домысливай; если пункта нет, напиши «нет».

Запись:
{transcript}"""

res = api("POST", "/assistant/llm", json={"run_id": RUN, "prompt": PROMPT, "source": "raw"})
print(f"{res['model']} | {res['duration_ms'] / 1000:.1f} с\n")
print(res["text"])

### Много записей одним своим промптом
Каждая запись: загрузка → транскрибация → ваш промпт → `data/output/<имя>.custom.txt`. Уже обработанные пропускаются.
Список `FILES` — пути к записям на вашем диске (или оставьте пустым и возьмите всё из `data/audio`).

In [ ]:
OUT = ROOT / "data" / "output"
OUT.mkdir(parents=True, exist_ok=True)
FILES = []                                    # например [Path("~/Downloads/rec1.wav").expanduser()]
names = [f.name for f in FILES] or [f["file_name"] for f in api("GET", "/assistant/audio")]

for path in FILES:                            # загрузка (уже загруженные пропускаются: 409)
    with open(path, "rb") as fh:
        r = requests.post(BASE + "/assistant/audio", files={"file": (path.name, fh)}, timeout=1800)
    print("upload", path.name, r.status_code)

for name in names:
    target = OUT / f"{Path(name).stem}.custom.txt"
    if target.exists():
        print("пропуск", name); continue
    run = api("POST", "/assistant/transcribe", json={"file_name": name})["run_id"]
    answer = api("POST", "/assistant/llm", json={"run_id": run, "prompt": PROMPT})["text"]
    target.write_text(answer, encoding="utf-8")
    print("готово", target.name, f"(run_id={run})")

## B10. Сводка за период в Excel
Например, все дейлики за неделю: что сделано и что запланировано, одной таблицей.
`POST /assistant/digests` для каждой встречи периода делает запрос к модели с промптом `prompts/digests/<name>/`
(он возвращает строки таблицы), склеивает строки и пишет `data/output/digests/<id>_<имя>_<от>_<до>.xlsx`.
Колонки задаёт `prompts/digests/<name>/digest.yaml`, инструкции модели — `system.md` и `user.md` рядом.
Период по умолчанию — последние 7 дней; встречи отбираются по типу (`kinds`), тегам и датам.

In [ ]:
import requests, time
from datetime import date, timedelta

DIGEST = "weekly_dailies"                     # папка в prompts/digests/
DATE_TO = date.today()
DATE_FROM = DATE_TO - timedelta(days=6)       # последняя неделя; можно задать конкретные даты date(2026, 11, 2)

body = {"name": DIGEST, "date_from": DATE_FROM.isoformat(), "date_to": DATE_TO.isoformat(), "wait": False,
        # "kinds": ["daily"],                 # по умолчанию берётся из digest.yaml
        # "tags": ["проект 3924"],            # встреча должна иметь все эти теги
        # "run_ids": [1, 2, 3],               # вместо периода: явный список встреч
        # "source": "raw",                    # по умолчанию из digest.yaml
        }
d = api("POST", "/assistant/digests", json=body)
DIGEST_ID = d["id"]
print("сводка", DIGEST_ID, "| встреч в выборке:", d["total"])

In [ ]:
# ход построения: на каждую встречу ~десятки секунд на локальной модели; Ctrl+C останавливает только ожидание
while True:
    d = requests.get(f"{BASE}/assistant/digests/{DIGEST_ID}", timeout=60).json()
    print(f"\r{d['status']}: встреч {d['done']}/{d['total']}, строк {d['rows_count']}      ", end="")
    if d["status"] != "running":
        break
    time.sleep(3)
print()
if d["status"] == "error":
    raise RuntimeError(d["error"])
display(pd.DataFrame(d["meetings"]))                       # какая встреча обработана, какая нет и почему
df = pd.DataFrame(d["rows"], columns=d["headers"])         # headers сохраняет порядок колонок
display(df)

In [ ]:
# Excel: скачать из сервиса (работает и когда сервис в Docker) и открыть
xlsx = requests.get(f"{BASE}/assistant/digests/{DIGEST_ID}/xlsx", timeout=60)
xlsx.raise_for_status()
target = ROOT / "data" / "output" / "digests" / f"digest_{DIGEST_ID}.xlsx"
target.parent.mkdir(parents=True, exist_ok=True)
target.write_bytes(xlsx.content)
print(target)
# import subprocess; subprocess.run(["open", str(target)])      # macOS: открыть в Excel

### Список и удаление сводок
Строки сводок хранятся в Postgres (таблица `digests`), поэтому их можно смотреть через `table("digests")` из B7a.
Удаление убирает строку из Postgres и xlsx-файл на диске; сначала пробный прогон.

In [ ]:
display(pd.DataFrame(api("GET", "/assistant/digests")))

def delete_digest(digest_id=None, *, everything=False, confirm=False):
    params = {"dry_run": not confirm, **({"all": True} if everything else {})}
    path = "/assistant/digests" if everything else f"/assistant/digests/{digest_id}"
    result = api("DELETE", path, params=params)
    print("УДАЛЕНО" if confirm else "пробный прогон (ничего не удалено)", "->", result)

delete_digest(DIGEST_ID)                    # confirm=True — удалить по-настоящему
# delete_digest(everything=True)            # все сводки

## B11. Разбор картинок (независимо от встреч)
Фото + ваш промпт → текст модели (`qwen3.5:4b` умеет vision). Со встречами это не связано: свои файлы (`data/images/`),
свои промпты (`prompts/images/<имя>/system.md` и `user.md`, метки `{file}`, `{count}`, `{date}`, `{tags}`), свои записи в Postgres (`image_analyses`).
Перед отправкой фото поворачивается по EXIF и уменьшается до `ASSISTANT_IMAGE_MAX_SIDE` (1568 px), так что снимок с телефона
в 12 Мп подходит как есть (формат HEIC нужно конвертировать в JPEG).
Для маленькой модели: печатный текст она читает хорошо, мелкий, рукописный и плотные таблицы хуже; имена и числа проверяйте глазами.

In [ ]:
from pathlib import Path

PHOTOS = []        # пути к фото на вашем диске, например [Path("~/Downloads/доска.jpg").expanduser()]

show_images(PHOTOS)   # превью до загрузки: с поворотом и уменьшением, как увидит модель
if PHOTOS:         # загрузка в data/images (занятое имя: 409, добавьте overwrite=true, чтобы заменить)
    files = [("files", (p.name, open(p, "rb"), "image/jpeg")) for p in PHOTOS]
    show(api("POST", "/assistant/images", files=files, data={"overwrite": "false"}))

images = api("GET", "/assistant/images")
display(pd.DataFrame(images))
print("промпты:", api("GET", "/assistant/prompts")["image_prompts"])

In [ ]:
show_images([settings.images_dir / n for n in IMAGES])   # что уйдёт в модель

# Все выбранные фото одним запросом (несколько снимков одной доски/страницы); результат один
IMAGES = [images[0]["file_name"]] if images else []
res = api("POST", "/assistant/images/analyze", json={
    "files": IMAGES,
    "prompt_name": "tasks_from_photo",          # папка prompts/images/<имя>/ ...
    # "prompt": "Опиши, что на фото. Файл: {file}",   # ... или свой текст вместо prompt_name (и "system": "...")
    "tags": ["тест"],
    "mode": "together",
})
for item in res:
    print(f"{item['files']} | {item['model']} | {item['duration_ms'] / 1000:.1f} с | id={item['analysis_id']}")
    print(item["error"] or item["text"])

### Пачка фото одним промптом
`mode="each"`: отдельный запрос и отдельная запись в Postgres на каждое фото, сбой одного не останавливает остальные.
Результаты пишутся ещё и в `data/output/images/<файл>.txt`.

In [ ]:
show_images([settings.images_dir / f["file_name"] for f in images], width=200)   # что уйдёт в модель

OUT_IMAGES = ROOT / "data" / "output" / "images"
OUT_IMAGES.mkdir(parents=True, exist_ok=True)

batch = api("POST", "/assistant/images/analyze", json={
    "files": [f["file_name"] for f in images],            # все загруженные; можно выбрать часть
    "prompt_name": "extract_text",
    "mode": "each",
})
for item in batch:
    name = item["files"][0]
    if item["error"]:
        print("ОШИБКА", name, item["error"])
        continue
    (OUT_IMAGES / f"{Path(name).stem}.txt").write_text(item["text"], encoding="utf-8")
    print("готово", name, f"({item['duration_ms'] / 1000:.1f} с)")

### История разборов и удаление
Файлы фото в `data/images` остаются при удалении записи о разборе. Сначала пробный прогон.

In [ ]:
history = api("GET", "/assistant/images/analyses", params={"limit": 100})        # + tag=..., file_name=...
display(pd.DataFrame(history, columns=["id", "images", "prompt_name", "tags", "model", "duration_ms", "text", "created_at"]))

def delete_image_analysis(analysis_id=None, *, everything=False, confirm=False):
    params = {"dry_run": not confirm, **({"all": True} if everything else {})}
    path = "/assistant/images/analyses" if everything else f"/assistant/images/analyses/{analysis_id}"
    result = api("DELETE", path, params=params)
    print("УДАЛЕНО" if confirm else "пробный прогон (ничего не удалено)", "->", result)

if history:
    delete_image_analysis(history[0]["id"])          # confirm=True — удалить по-настоящему
# delete_image_analysis(everything=True)             # все разборы картинок

# Часть C. Prompt Studio: создать, проверить, сравнить промпты
Работаете как промпт-инженер, код не трогаете:
1. **Создать** промпт из шаблона (или скопировать похожий): формой или строкой `lab.new(...)`.
2. **Править** файлы в `prompts/...` в любом редакторе (заготовки `<...>` заменить своим текстом).
3. **Проверить**: пустые файлы, опечатки в метках `{transcrip}`, забытые заготовки.
4. **Сравнить** версии на реальной встрече/фото: результаты рядом, время, без записи в Postgres.
5. Победитель уже лежит в файле. Для типа встречи достаточно, чтобы имя папки совпало с ключом типа (или `prompt:` в `config/meeting_kinds.yaml`).

Четыре вида: `audience` (статус сотрудника для руководства/РП/команды), `meeting` (разбор встречи по типу), `digest` (сводка в Excel), `image` (фото).
Править и удалять тоже можно формой ниже (удалённое уходит в корзину, правки хранят историю).
Файлы читаются на каждый запрос: перезапуск сервиса и ядра не нужен. То же из терминала: `python scripts/prompt_studio.py --help`.

In [ ]:
import difflib, html
import pandas as pd
from IPython.display import HTML
from audio_assistant.client.prompt_lab import PromptLab, PromptTrial, transcript_from_run

lab = PromptLab(settings.prompts_dir, settings.kinds_file,
                protected=(("audience", settings.default_audience),))   # аудиторию по умолчанию не удалить
trial = PromptTrial(pipeline)

display(pd.DataFrame(lab.catalog()))                    # все промпты
checks = pd.DataFrame(lab.check_all())
display(checks[checks["уровень"] != "info"] if len(checks) else "замечаний нет")

## C1. Новый промпт
`lab.new(вид, имя)` копирует шаблон из `prompts/_templates/<вид>/`. Для `meeting` тип встречи сам добавляется в `config/meeting_kinds.yaml`.
`copy_from="..."` берёт за основу уже работающий промпт, и его правят, а не пишут с нуля. Имя: латиница, цифры, `_`.
Это то же, что кнопка «Создать» в форме ниже. Править: `lab.read/save`, удалять: `lab.delete(..., confirm=True)`, вернуть: `lab.undelete(lab.trash()[0])`.

In [ ]:
# Раскомментируйте нужное. Существующее не перезаписывается.
# lab.new("meeting",  "retro", title="Ретроспектива")                 # новый тип встречи + разбор
# lab.new("meeting",  "daily_v2", copy_from="daily")                  # вариант существующего для сравнения
# lab.new("audience", "investors", copy_from="management")           # новая аудитория
# lab.new("digest",   "one_to_one_month")                             # новая сводка в Excel
# lab.new("image",    "receipt")                                      # новый разбор фото


## C2. Проверка и итоговый текст
`check` ловит ошибки до запуска. `preview` показывает, что реально уйдёт модели (system + user с подставленными значениями).

In [ ]:
KIND_, NAME_ = "meeting", "daily"
for issue in lab.check(KIND_, NAME_):
    print(issue)
print()
print(lab.preview(KIND_, NAME_, transcript="Иван: вчера сделал выгрузку, сегодня обучу модель."))

## C3. Материал для проб
Берём реальную встречу из Postgres (`RUN` из B2) с уже размеченными говорящими. Если сервис не запущен, подойдёт `attributed` из части A.

In [ ]:
try:
    DETAIL = api("GET", f"/assistant/runs/{RUN}")
    SAMPLE = transcript_from_run(DETAIL)
    SAMPLE_RUN = DETAIL["run"]
except Exception:
    SAMPLE, SAMPLE_RUN = attributed, None
print("реплик:", len(SAMPLE.segments), "| слов:", len(SAMPLE.text.split()))

def compare(results, width=48):
    """Результаты версий промпта рядом + время. Ошибка одной версии не мешает остальным."""
    cells = "".join(
        f"<td style='vertical-align:top;width:{width}%;padding:8px;border:1px solid #8884'>"
        f"<b>{html.escape(r.name)}</b> <small>{r.seconds:.1f} с</small><hr>"
        f"<pre style='white-space:pre-wrap'>{html.escape(r.error and 'ОШИБКА: ' + r.error or r.text)}</pre></td>"
        for r in results)
    display(HTML(f"<table style='width:100%'><tr>{cells}</tr></table>"))
    if len(results) == 2 and all(not r.error for r in results):
        a, b = (r.text.splitlines() for r in results)
        print("строк отличается:", sum(1 for d in difflib.ndiff(a, b) if d[0] in "+-"))

## Форма: создать, править, сравнить, удалить (без кода)
Выберите вид и промпт, правьте файлы во вкладках, **Сохранить** (прежний текст уходит в историю, **Откатить** возвращает), **Проверить**,
**Итоговый текст** (что уйдёт модели), **Пробный запуск** на встрече из C3 (в «Сравнить» можно перечислить версии через запятую: `daily, daily_v2`).
**Создать новый**: имя латиницей, основа «(шаблон)» или любой существующий промпт. Для `meeting` в поле «Название» пишется имя типа встречи.
**Удаление** уходит в корзину `prompts/.trash/` и возвращается кнопкой «Вернуть». Для разбора встречи, привязанного к типу, нужна отметка «и убрать тип встречи».
Нужен `ipywidgets` (`pip install -e ".[dev]"`).

In [ ]:
from audio_assistant.client.prompt_lab_ui import PromptStudioUI

ui = PromptStudioUI(lab, trial=trial, sample=lambda: SAMPLE, run=lambda: SAMPLE_RUN, day=lambda: DAY)
ui.show()

## C4. Сравнение версий
Список имён = версии одного промпта (`daily` и `daily_v2`). Меняете файл, запускаете ячейку снова.

In [ ]:
# разбор встречи: любое число версий; тип/дата/теги берутся из запуска
compare(trial.meeting(SAMPLE, ["daily", "other"], run=SAMPLE_RUN))

In [ ]:
# аудитории (статус по каждому сотруднику): те же реплики, разные адресаты
compare(trial.audience(SAMPLE, ["team", "management"], day=DAY))

In [ ]:
# фото: файлы из data/images, разные промпты
# compare(trial.image([images[0]["file_name"]], ["describe", "extract_text"]))

## C5. Как оптимизировать промпт для маленькой модели (4B)
- **Формат ответа покажите шаблоном**, а не словами: готовые строки `- Сделано: ...` работают лучше, чем «ответь структурированно».
- **Запреты конкретные**: «не пиши "не указано"» лучше, чем «не выдумывай». Если фраза-паразит появилась, добавьте её в запрет дословно.
- **Один промпт = одна задача.** Нужно и протокол, и задачи: два промпта (`meeting`) и два прогона.
- **Короткий system** (до ~1500 символов). Длинные инструкции размываются; `check` предупредит после 6000.
- **Пример в `examples.md`** весит больше, чем абзац правил. Берите настоящие хорошие ответы и убирайте чужие факты.
- **Меняйте одно за раз** и сравнивайте с прошлой версией в C4 на тех же репликах. Старую версию оставляйте как `имя_v1`.
- Имена людей искажаются Whisper: список участников есть в метке `{participants}`, можно попросить сверять имена с ним.